# Three Problems at Swineboils
### Margins, networks, and the Allocation Helm

A new term is about to begin at Swineboils College of Spellcraft and Sorcery.
The corridors are filling with floating trunks, the cauldrons are warming up,
and the Headmistress is trying to get everything ready before the students arrive.
Unfortunately, a few of the school's old enchantments have become unreliable.

You have joined the school to help its staff learn from the records they have
collected over the years. On your first morning, three requests arrive on your
desk: the alchemy department has troublesome potion batches, the library's rune
reader is making mistakes, and the admissions office needs help assigning Houses.

Each task stands on its own; complete setup, then start the task you are working on.


In [ ]:
# RUN ONLY
from pathlib import Path
import sys
import numpy as np
import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, mean_squared_error

BASE = next((p for p in (Path.cwd(), Path.cwd() / "lecture2")
             if (p / "data" / "potions.npz").exists()), None)
if BASE is None:
    raise FileNotFoundError("Open this notebook from the project root or lecture2 directory.")
if str(BASE.resolve()) not in sys.path:
    sys.path.insert(0, str(BASE.resolve()))

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
if "scorecard" not in globals():
    scorecard = {}


## 1. Unstable potions

The alchemy department needs a classifier that labels a batch **stable (0)** or
**unstable (1)** from six instrument readings. Build a classifier, evaluate it,
and justify which model the department should deploy.

All model and hyperparameter choices are yours. Try a few different network architectures and compare their results before choosing your final model.

Deliver your model, training and validation accuracy, and a brief justification.
For final evaluation, provide `predict_potions(raw_X)`, returning one 0/1 label
per row. The argument has the same columns and units as the supplied data.


In [ ]:
# RUN ONLY
POTION_FEATURES = ["crystal_resonance", "cauldron_pressure", "vapour_index",
                   "mana_density", "silver_trace", "colour_shift"]
with np.load(BASE / "data" / "potions.npz", allow_pickle=False) as data:
    X_potion_train = data["X_train"].copy()
    y_potion_train = data["y_train"].copy()
    X_potion_val = data["X_val"].copy()
    y_potion_val = data["y_val"].copy()

print("Training:", X_potion_train.shape, "Validation:", X_potion_val.shape)
display(pd.DataFrame(X_potion_train, columns=POTION_FEATURES).head())


In [ ]:
# TODO
# Build and evaluate your classifier here. Add cells as needed.

def predict_potions(raw_X):
    """Return labels with shape (len(raw_X),)."""
    raise NotImplementedError("Connect your final classifier here.")

potion_ready = False  # Set True only when your final choice is made.


**TODO — deployment decision:** Which model should be deployed, and what evidence
supports that choice beyond training accuracy?

Write your answer here.

In [ ]:
# RUN ONLY
if not potion_ready:
    print("Finish choosing your model before the final evaluation.")
elif "Potions" not in scorecard:
    with np.load(BASE / "data" / "potions.npz", allow_pickle=False) as data:
        predictions = np.asarray(predict_potions(data["X_test"]))
        labels = data["y_test"]
    assert predictions.shape == labels.shape and np.isin(predictions, [0, 1]).all()
    scorecard["Potions"] = {"task": "Potions",
                           "test_accuracy": float(np.mean(predictions == labels))}
if "Potions" in scorecard:
    display(pd.Series(scorecard["Potions"]))


## 2. The library rune reader

The library uses handwritten runes to label its shelves. Lately, its reader has
been sending students to the wrong books. The librarian has handed you the current
network and a small allowance to improve it.

Train the starter and measure its performance. Then build a replacement that
reaches **at least 90% validation accuracy at the lowest cost**, with a maximum
of **25 coins**. All versions must learn from the same training set. The test
set is reserved for one final report.


In [ ]:
# RUN ONLY
_digits = load_digits()
_dev_ids, _test_ids = train_test_split(np.arange(len(_digits.target)), test_size=0.2,
                                     stratify=_digits.target, random_state=2022)
_train_ids, _val_ids = train_test_split(_dev_ids, test_size=0.25,
                                      stratify=_digits.target[_dev_ids], random_state=2023)
rune_train_images = _digits.images[_train_ids].copy()
rune_train_labels = _digits.target[_train_ids].copy()
rune_val_images = _digits.images[_val_ids].copy()
rune_val_labels = _digits.target[_val_ids].copy()

print("Training images:", rune_train_images.shape, "Validation images:", rune_val_images.shape)
fig, axes = plt.subplots(2, 8, figsize=(10, 4), layout="constrained")
for ax, picture, label in zip(axes.flat, rune_train_images[:16], rune_train_labels[:16]):
    ax.imshow(picture, cmap="gray", vmin=0, vmax=16, interpolation="nearest")
    ax.set_title(str(label))
    ax.axis("off")
fig.suptitle("Library scans and their correct rune labels")
plt.show()
del _digits, _dev_ids, _test_ids, _train_ids, _val_ids


In [ ]:
# RUN ONLY
torch.manual_seed(7)
starter_model = nn.Sequential(
    nn.Linear(64, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 10),
)
for layer in starter_model:
    if isinstance(layer, nn.Linear):
        nn.init.normal_(layer.weight, mean=0.0, std=0)
        nn.init.zeros_(layer.bias)

print(starter_model)


### Training and the upgrade allowance

Write the training code and measure its performance.

For your replacement, the model and training choices are yours. Keep a record
of your experiments and the resources used to train your final model.

| Resource | Cost in coins |
|---|---:|
| Each 1,000 parameters, including biases | 1 (fractional) |
| Each hidden layer using leaky ReLU | 3 |
| Each hidden layer using ReLU | 2 |
| Each residual connection | 2 |
| Initialization: zeros / normal (std != 0) | 0 / 2 |
| Each training epoch | 0.1 |

Use one of these initialization choices for your linear layers. Cost is for
training the submitted model from scratch; exploratory runs are free. Count all
epochs used to train that model, including any training resumed later.

The score is **100 × test accuracy / (1 + cost)**, eligible only when validation
accuracy is at least 90% and cost is at most 25. Higher is better.

Deliver the starter's measured performance, your replacement, and evidence for
what changed. For final evaluation, provide `predict_runes(raw_images)`, returning
one digit label per image. It should use your final model.


In [ ]:
# RUN ONLY
def rune_cost(model, epochs, leaky_layers=0, relu_layers=0, residual_connections=0,
              initialization="zeros"):
    parameters = sum(p.numel() for p in model.parameters())
    return (parameters / 1000 + 3 * leaky_layers + 2 * relu_layers
            + 2 * residual_connections
            + {"zeros": 0, "normal": 2}[initialization] + 0.1 * epochs)


In [ ]:
# TODO
# Prepare the data, train the starter, and run your experiments. Add cells as needed.
# Assign your chosen trained network to final_rune_model.
final_rune_model = None

def predict_runes(raw_images):
    """Return digit labels with shape (len(raw_images),)."""
    raise NotImplementedError("Use your final model here.")

# Fill in the resources used by your final model.
final_epochs = None
final_leaky_layers = None
final_relu_layers = None
final_residual_connections = None
final_initialization = None  # "zeros" or "normal" (nonzero standard deviation)
runes_ready = False


**TODO — repair report:** Diagnose the starter using your evidence. Which change
improved generalization per unit cost, rather than only training performance?

Write your answer here.

In [ ]:
# RUN ONLY
if not runes_ready:
    print("Finish choosing and training your model before the final evaluation.")
elif "Rune reader" not in scorecard:
    assert isinstance(final_rune_model, nn.Module)
    assert isinstance(final_epochs, int) and final_epochs > 0
    assert isinstance(final_leaky_layers, int) and final_leaky_layers >= 0
    assert isinstance(final_relu_layers, int) and final_relu_layers >= 0
    assert isinstance(final_residual_connections, int) and final_residual_connections >= 0
    assert final_initialization in ("zeros", "normal")
    cost = rune_cost(final_rune_model, final_epochs,
                     leaky_layers=final_leaky_layers, relu_layers=final_relu_layers,
                     residual_connections=final_residual_connections,
                     initialization=final_initialization)
    final_rune_model.eval()
    with torch.no_grad():
        predictions = np.asarray(predict_runes(rune_val_images))
    assert predictions.shape == rune_val_labels.shape and np.isin(predictions, np.arange(10)).all()
    validation_accuracy = float(np.mean(predictions == rune_val_labels))
    if validation_accuracy < .90 or cost > 25:
        print(f"Not eligible yet: validation={validation_accuracy:.2%}, cost={cost:.3f}")
    else:
        digits = load_digits()
        _, test_ids = train_test_split(np.arange(len(digits.target)), test_size=0.2,
                                      stratify=digits.target, random_state=2022)
        with torch.no_grad():
            predictions = np.asarray(predict_runes(digits.images[test_ids]))
        labels = digits.target[test_ids]
        assert predictions.shape == labels.shape and np.isin(predictions, np.arange(10)).all()
        test_accuracy = float(np.mean(predictions == labels))
        scorecard["Rune reader"] = {"task": "Rune reader",
            "validation_accuracy": validation_accuracy, "test_accuracy": test_accuracy,
            "cost": cost, "efficiency_score": 100 * test_accuracy / (1 + cost)}
        del digits, test_ids
if "Rune reader" in scorecard:
    display(pd.Series(scorecard["Rune reader"]))


## 3. The Allocation Helm

You are the school's enchanted House allocator. Your judgement has deteriorated
with age. A temporary restoration gives you one chance to assign this year's
20 students. Use the historical records to make the class's average **Ofspev
rating** as high as possible. This rating measures a student's eventual impact.
You may assign any number of students to each House.

Use the records to make and justify your allocations, supported by an evaluation
you consider defensible.

Submit a table indexed by incoming student ID, with columns `House` and
`Predicted rating`. Report its predicted average and explain your evaluation.
No actual incoming-student ratings are available before your choices are frozen.

Adapted from abstractapplic's
[D&D.Sci September 2022: The Allocation Helm](https://www.lesswrong.com/posts/DKDDT8hGCTz8AxBQF/d-and-d-sci-september-2022-the-allocation-helm).
The original CSVs are included locally. The linked comments contain other players'
answers; save those and the published solution for after your submission.


In [ ]:
# RUN ONLY
helm_history = pd.read_csv(BASE / "data" / "dset.csv", index_col=0)
helm_history.index.name = "Record"
helm_incoming = pd.read_csv(BASE / "data" / "incoming_class.csv", index_col="Student")
HOUSES = ("Dragonslayer", "Thought-Talon", "Serpentyne", "Humblescrumble")
print("Historical records:", helm_history.shape, "Incoming students:", helm_incoming.shape)
display(helm_history.head())
display(helm_incoming)


| Field | Meaning |
|---|---|
| Record / Student | Historical record ID / incoming student ID |
| Intellect, Integrity, Courage, Reflexes, Patience | Five numeric readings made by the Helm |
| House | Historical allocation; one of the four Houses |
| Ofstev Rating | Recorded outcome (the CSV uses this spelling of Ofspev) |
| Year | Intake year |

In [ ]:
# TODO
# Investigate, model, and evaluate here. Add cells as needed.
# Create allocation: a DataFrame indexed by student ID, with House and Predicted rating columns.
allocation = None
helm_ready = False  # Set True to freeze your completed allocation.


**TODO — allocation report:** Explain your approach and its evaluation. State your
predicted class average and the main limitation of your evidence.

Write your answer here.

In [ ]:
# RUN ONLY
if not helm_ready:
    print("Finish your allocation before continuing.")
elif "frozen_allocation" not in globals():
    assert isinstance(allocation, pd.DataFrame)
    assert not allocation.index.has_duplicates
    assert set(allocation.index) == set(helm_incoming.index)
    assert allocation["House"].isin(HOUSES).all()
    assert np.isfinite(allocation["Predicted rating"].to_numpy(dtype=float)).all()
    frozen_allocation = allocation.loc[helm_incoming.index, ["House", "Predicted rating"]].copy(deep=True)
if "frozen_allocation" in globals():
    display(frozen_allocation)
    print("Frozen predicted average:", frozen_allocation["Predicted rating"].mean())


---
## SPOILER / FINAL EVALUATION — Allocation Helm

**Stop here until your allocation and report are final.** The next section loads
the true scoring rule and reveals the achieved expected class rating. Do not use
this feedback to revise your submission. The scoring source is in
`_helm_reveal.py`; leave it closed until you are ready for the reveal.

In [ ]:
# TODO
reveal_helm = False  # Set True only after freezing your allocation and writing your report.


In [ ]:
# RUN ONLY
if not reveal_helm:
    print("Scoring rule remains sealed.")
elif "frozen_allocation" not in globals():
    print("Freeze your allocation first.")
else:
    if "helm_final_result" not in globals():
        from _helm_reveal import reveal
        actual_incoming = pd.read_csv(BASE / "data" / "incoming_class.csv", index_col="Student")
        helm_final_result = reveal(frozen_allocation, actual_incoming)
        scorecard["Allocation Helm"] = {"task": "Allocation Helm",
            "class_rating": helm_final_result["class_rating"],
            "predicted_rating": float(frozen_allocation["Predicted rating"].mean())}
    display(pd.Series({k: v for k, v in helm_final_result.items() if k != "table"}))
    display(helm_final_result["table"])


## Your final scorecard

Each completed task contributes its final result. Blank entries mean that a task
has not been submitted. Save your notebook with the results and written decisions.

In [ ]:
# RUN ONLY
if scorecard:
    display(pd.DataFrame.from_dict(scorecard, orient="index").drop(columns="task"))
else:
    print("No final submissions yet.")
